# Notebook summary

**Principle.** The network predicts each time bin from the last $D$ bins of activity only (the spiking motif window), $s(t) = f\big(s(t-D), \dots, s(t-1)\big)$, where $D$ = `num_delay` (41 bins of 1 ms) and $s^*$ = `frozen_target` is the motif of $T$ = `N_time` bins (1000 bins). The motif is made periodic by copying its first $D$ bins onto its last $D$ bins:

$$ s^*(t) = s^*(t - P) \quad \text{for } T - D \le t < T, \qquad P = T - D \ (959 \text{ bins}). $$

Training on a single motif covers the $P$ transitions $t \in [D, T)$, which are exactly all the transitions of a cycle of period $P$, including the one that wraps the end of the motif back onto its beginning. Once the network reaches the end of the motif, the window of its last $D$ bins is therefore identical to the trigger window, so it replays $s^*(D), s^*(D+1), \dots$ again: the polychronous chain closes on itself and the motif becomes a self-sustained cycle. Retrieval is then tested over `N_repeat` = 30 consecutive periods after a single trigger window, and the noise sweep measures, through the F1 score of each of `N_period` = 100 periods of free replay, whether the cycle survives or fades.

This notebook trains MNESIS/HD-SNN on periodic spike patterns and evaluates robustness to input perturbations.

## Pipeline
1. Load the shared setup from `mnesis_boilerplate` and `mnesis_chains`.
2. Build `PeriodicSpikingPattern`: Bernoulli target (`p_A`) with periodic boundary copy over `num_delay` bins;
   as a `StochasticSpikingPattern`, each call returns a balanced-flip realisation (`p_flip`).
3. Instantiate `Params` + `HD_SNN`, then load or train and cache `{datetag}_{exp}.pth` (with lock file).
4. Evaluate one-period retrieval (precision, recall, F1) and plot raster diagnostics.
5. Evaluate long-horizon retrieval over repeated periods (`N_repeat`).
6. Same long-horizon retrieval in a second cache (the `flip_bits(..., 0.323)` perturbation is
   commented out, so the trigger is clean).
7. Sweep perturbation level (`p_flip` in [0,1], balanced flips of the motif as in notebook 14,
   seeded): one noisy trigger, then `N_period` periods (of `N_time - num_delay` bins) of free
   replay, each scored (F1) against the frozen target (`scores_np`, shape `(N_scan, N_period)`).

## Main outputs
- Model checkpoint: `{datetag}_{exp}.pth`.
- Optional cached spikes: clean/noisy periodic retrieval tensors.
- Noise-sweep cache: `{datetag}_{exp}-with-noise_N_period={N_period}.npz` (`p_flip`, `scores_np`).
- Metrics and plots: precision/recall/F1, rasters, and F1-vs-period trajectories.

In [1]:
from mnesis_boilerplate import torch, np, plt, splt, data_cache, RECOMPUTE, Params
from mnesis_boilerplate import printfig, get_scores, get_f1score, trange
from mnesis_chains import HD_SNN, StochasticSpikingPattern
# RECOMPUTE = True
exp = 'periodic'

datetag = '2026-09-07'
Using device: mps


## working memory with periodic targets

Setting up:

In [2]:
# (p_flip) as notebook 11; the periodic wrap is restored after each realisation.
class PeriodicSpikingPattern(StochasticSpikingPattern):
    def __init__(self):
        super().__init__()
        self.desc = "Periodic pattern generator"
        self.is_periodic = True

    def init(self, opt, verbose=False):
        self.opt = opt
        frozen_target_generator = torch.Generator()  # used once to generate the target pattern
        frozen_target_generator.manual_seed(opt.seed)
        p_bias = opt.p_A * torch.ones((opt.N_pattern, opt.N_neuron, opt.N_time))
        self.frozen_target = torch.bernoulli(p_bias, generator=frozen_target_generator)
        self.frozen_target = self.frozen_target.float()
        self.frozen_target = self.frozen_target.to(opt.device)
        if verbose:
            print(f"Target pattern generated with shape {self.frozen_target.shape} and mean {self.frozen_target.mean().item():.3e}")

        # Make the target periodic by copying the first opt.num_delay time steps to the end of the sequence
        # such that the polychronous chain is virtually endless
        self.frozen_target[:, :, (-opt.num_delay):] = self.frozen_target[:, :, :opt.num_delay]

    def __call__(self, seed=None, verbose=False):
        realisation = super().__call__(seed=seed, verbose=verbose)
        # keep the chain endless: the last num_delay bins repeat the first ones
        realisation[:, :, (-self.opt.num_delay):] = realisation[:, :, :self.opt.num_delay]
        return realisation


In [3]:
opt = Params()
hd = HD_SNN(opt, PeriodicSpikingPattern())
hd.net.to(hd.opt.device)
hd.pattern_object().shape, opt

(torch.Size([16, 1024, 1000]),
 Params(datetag='2026-09-07', N_neuron=1024, num_delay=41, N_pattern=16, N_time=1000, N_pretime=50, p_A=0.00016, p_flip=0.01, seed=2026, lif_beta=0.7, lif_threshold=0.75, learn_beta=False, learn_threshold=False, do_pinv=True, do_deconv=True, num_epochs=256, num_warmup_epochs=16, base_lr=0.04, final_lr=0.0004, delta1=0.003, delta2=1e-05, dropout=0.6, alpha_surrogate=5.0, surrogate_name='FastSigmoid', loss_name='SpikeF1scoreLoss', reset_mechanism='zero', optimizer='adadelta', verbose=False, fig_width=15.92, phi=1.61803, fig_height=9.839125356142963, N_time_show=1000, N_neuron_show=1024, i_pattern=0, N_scan=14, N_cv=11))

## Learn the periodic target

In [ ]:
model_filename = data_cache / f"{hd.opt.datetag}_{exp}.pth"
lock_filename = model_filename.with_suffix('.lock')
if RECOMPUTE:
    model_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    model_state_dict = torch.load(model_filename, map_location=torch.device(hd.opt.device))
    hd.net.load_state_dict(model_state_dict)
    hd.net.eval()
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Model weights loaded from {model_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Model file not found: {model_filename}, inititializing and learning the new model.")
        lock_filename.touch(exist_ok=True)
        ##################
        hd.update_weight()
        hd.learn_model()
        ##################
        torch.save(hd.net.state_dict(), model_filename)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Model file is locked: {lock_filename}, passing.")

Model file not found: ../cached_data/2026-09-07_periodic.pth, inititializing and learning the new model.


In [ ]:
with torch.no_grad():
    target = hd.pattern_object.frozen_target # noise-free reference, scored against
    cue = hd.pattern_object(seed=hd.opt.seed) # seeded noisy realisation, provides the trigger
    target_full = torch.nn.functional.pad(target, (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(target=cue, seed=hd.opt.seed)
    _, _, spikes = hd.forward_pass(input_spikes)
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(hd.opt.N_time+hd.opt.N_pretime)]
    target_evoked = target[:, :, hd.opt.num_delay:]

    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
fig, ax = plt.subplots()
splt.raster(spikes[opt.i_pattern, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[opt.i_pattern, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[opt.i_pattern, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
spikes.shape, spikes[opt.i_pattern, :, :].sum().item(), target[opt.i_pattern, :, :].sum().item()

## Retrieve a periodic memory

In [ ]:
N_repeat = 30

In [ ]:
target.shape

In [ ]:
spikes_filename = data_cache / f"{hd.opt.datetag}_{exp}_spikes.pth"
lock_filename = spikes_filename.with_suffix('.lock')

if RECOMPUTE:
    spikes_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
with torch.no_grad():    
    target_full = torch.nn.functional.pad(target[:, :, :(-hd.opt.num_delay)].repeat(1, 1, N_repeat), (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(target=target, N_time=N_repeat*(opt.N_time-opt.num_delay))
    try:
        spikes = torch.load(spikes_filename, map_location=torch.device(hd.opt.device))
        lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
        print(f"Model weights loaded from {spikes_filename}") # Add a log message
    except FileNotFoundError:
        if not lock_filename.exists():
            print(f"Spikes file not found: {spikes_filename}, running the model.")
            lock_filename.touch(exist_ok=True)
            ##################
            _, _, spikes = hd.forward_pass(input_spikes)
            ##################
            torch.save(spikes, spikes_filename)
            lock_filename.unlink(missing_ok=True)
        else:
            print(f"Model file is locked: {lock_filename}, passing.")
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(hd.opt.N_time+hd.opt.N_pretime)]
    target_evoked = target_full[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(hd.opt.N_time+hd.opt.N_pretime)]
    
    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
target_full.shape, input_spikes.shape, spikes.shape

In [ ]:
fig, ax = plt.subplots()
splt.raster(spikes[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
printfig(fig, 'periodic_spikes', fig_width=opt.fig_width, fig_height=opt.fig_height, figpath=None)
spikes.shape, spikes[opt.i_pattern, :, :].sum().item(), target[opt.i_pattern, :, :].sum().item()

## Retrieve a periodic memory with a little bit of noise

In [ ]:
spikes_filename = data_cache / f"{hd.opt.datetag}_{exp}_spikes_noisy.pth"
lock_filename = spikes_filename.with_suffix('.lock')
# RECOMPUTE = True
if RECOMPUTE:
    spikes_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE

with torch.no_grad():    
    target_full = torch.nn.functional.pad(target[:, :, :(-hd.opt.num_delay)].repeat(1, 1, N_repeat), (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(target=target, N_time=N_repeat*(opt.N_time-opt.num_delay))
    # input_spikes = flip_bits(input_spikes, .323, seed=hd.opt.seed)  # Add a seed for reproducibility
    try:
        spikes = torch.load(spikes_filename, map_location=torch.device(hd.opt.device))
        lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
        print(f"Model weights loaded from {spikes_filename}") # Add a log message
    except FileNotFoundError:
        if not lock_filename.exists():
            print(f"Spike file not found: {spikes_filename}, running the model.")
            lock_filename.touch(exist_ok=True)
            ##################
            _, _, spikes = hd.forward_pass(input_spikes)
            ##################
            torch.save(spikes, spikes_filename)
            lock_filename.unlink(missing_ok=True)
        else:
            print(f"Model file is locked: {lock_filename}, passing.")
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(hd.opt.N_time+hd.opt.N_pretime)]
    target_evoked = target_full[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(hd.opt.N_time+hd.opt.N_pretime)]
    
    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')


In [ ]:
target_full.shape, spikes.shape, input_spikes.shape

In [ ]:
fig, ax = plt.subplots()
splt.raster(spikes[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[opt.i_pattern, :opt.N_neuron_show, :].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
printfig(fig, 'periodic_spikes_noise', fig_width=opt.fig_width, fig_height=opt.fig_height, figpath=None)
spikes.shape, spikes[opt.i_pattern, :, :].sum().item(), target[opt.i_pattern, :, :].sum().item()

In [ ]:
fig, ax = plt.subplots()
ax.plot(spikes.mean(dim=[0, 1]).cpu(), alpha=.2)
# ax.plot(spikes[opt.i_pattern, ...].mean(dim=0).cpu(), alpha=.7)
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Firing rate (Hz)")
printfig(fig, 'periodic_firing', fig_width=opt.fig_width, fig_height=opt.fig_height, figpath=None)
spikes.shape, target_full.shape

## Retrieve a periodic memory with different levels of noise

In [ ]:
N_period = 100 # number of periods of free replay after the single noisy trigger

def get_scores_noise(N_scan=opt.N_scan, N_period=N_period):
    p_flip_default = hd.pattern_object.opt.p_flip
    p_flip = np.linspace(0, 1, N_scan, endpoint=True)
    # the last num_delay bins of the motif repeat the first ones, so the chain has this period (bins)
    period = hd.opt.N_time - hd.opt.num_delay
    # the evoked window starts at phase num_delay of the motif, so every period is scored
    # against the same slice as the one-period score above
    target_evoked = target[:, :, hd.opt.num_delay:]
    t_evoked = hd.opt.N_pretime + hd.opt.num_delay
    scores_np = np.zeros((N_scan, N_period))
    for i_scan in trange(N_scan):
        with torch.no_grad():
            # same noise as notebook 14: balanced flips of the motif (not of the whole input),
            # seeded; only the trigger window of this noisy realisation enters the network
            hd.pattern_object.opt.p_flip = p_flip[i_scan]
            cue = hd.pattern_object(seed=hd.opt.seed + i_scan)
            input_spikes = hd.get_input_spikes(target=cue, N_time=hd.opt.num_delay + N_period*period, seed=hd.opt.seed + i_scan)

            _, _, spikes = hd.forward_pass(input_spikes)

            for i_period in range(N_period):
                spikes_evoked = spikes[:, :, (t_evoked + i_period*period):(t_evoked + (i_period+1)*period)]
                scores_np[i_scan, i_period] = get_f1score(spikes_evoked, target_evoked).item()

    hd.pattern_object.opt.p_flip = p_flip_default
    return p_flip, scores_np

# N_period sets the length of the run, so it belongs to the cache key
npy_filename = data_cache / f"{hd.opt.datetag}_{exp}-with-noise_N_period={N_period}.npz"
lock_filename = npy_filename.with_suffix('.lock')
# RECOMPUTE = True
if RECOMPUTE:
    npy_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    data = np.load(npy_filename)
    p_flip = data['p_flip']
    scores_np = data['scores_np']
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Score weights loaded from {npy_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Score file not found: {npy_filename}, evaluating the score.")
        lock_filename.touch(exist_ok=True)
        ##################
        p_flip, scores_np = get_scores_noise()
        ##################        
        np.savez(npy_filename, p_flip=p_flip, scores_np=scores_np)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Score file is locked: {lock_filename}, passing.")

In [ ]:
if not lock_filename.exists():
    # the colorbar must be placed by the same layout engine as the one printfig applies
    fig, ax = plt.subplots(layout='constrained')
    # one colour per noise level, read on the colorbar rather than on a 14-entry legend
    cmap = plt.cm.viridis
    for i_scan in range(opt.N_scan):
        ax.plot(np.arange(1, N_period+1), scores_np[i_scan, :], color=cmap(p_flip[i_scan]))
    fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1)), ax=ax, label=r'$p_\mathrm{flip}$')
    ax.set_xlabel("Period number")
    ax.set_ylabel("F1 score")
    printfig(fig, 'periodic_scores', fig_width=opt.fig_width, fig_height=opt.fig_height)